# V15 · Datenanalyse mit pandas · Vertiefung zum Training

**Wo passt das Notebook hin?** Erst die Python-Werkstatt im Labor, dann das Training zu Hause (Browser, ca. 60 Min). Dieses Notebook ist die **Vertiefung** und läuft am besten im **Codespace** oder in Jupyter (dort sind pandas und matplotlib installiert). Es nutzt Python aus V01 bis V14 und neu pandas: `read_csv`, Spalten, Filtern mit einer Maske, `groupby`, `describe`.

**Die Messdatei erzeugt das Notebook selbst.** Die Werkzeug-Zelle unten legt `wellen.csv` im Ordner des Notebooks an: 250 Wellendurchmesser von zwei Drehmaschinen D1 und D2 (25 Stichproben mit je 5 Teilen je Maschine), Soll 20,000 mm, Toleranz ± 0,020 mm. Spalten: `stichprobe`, `maschine`, `schicht` (Früh bis Stichprobe 13, danach Spät), `teil`, `durchmesser_mm`. D1 läuft mittig und ruhig. D2 liegt etwas hoch und driftet ab Stichprobe 18 (Werkzeugverschleiß).

**Der Arbeitsablauf bei jeder Aufgabe:** (1) Welche Teilmenge ist gemeint (eine Maschine, eine Schicht)? (2) Maske bauen und filtern, (3) Kennzahl rechnen, (4) Ergebnis mit einer Überschlagsrechnung vergleichen, (5) die Prüfzelle laufen lassen. Die Prüfzellen rechnen die richtige Antwort unabhängig mit dem `csv`-Modul nach. Sie sind keine Musterlösungen, sondern zeigen nur, ob dein Ergebnis stimmt.

**Die Aufgaben bauen aufeinander auf:** Löse zuerst 1.2 (die Tabelle `df`), die späteren Aufgaben benutzen sie. Führe die Zellen von oben nach unten aus.

**Wichtig:** Bedingungen in Tabellen verknüpfst du mit `&` (und), `|` (oder), `~` (nicht), **jede in Klammern**. Mit `and`, `or`, `not` gibt es einen `ValueError`.

**Kleines Vokabular:** Ein **DataFrame** ist eine Tabelle, eine **Series** eine Spalte. Eine **Maske** ist eine Reihe aus True und False. **Cp** und **Cpk** messen die Prozessfähigkeit, die **Regelkarte** zeigt Stichprobenmittel mit **Eingriffsgrenzen** (nicht zu verwechseln mit den Toleranzgrenzen).

**Datenschutz:** Keine Firmen- oder Kundendaten in Prompts. Hier genügen Übungsdaten.

In [ ]:
# Werkzeug-Zelle: einmal ausführen, nicht verändern.
# Sie legt die Messdatei wellen.csv an und definiert die Selbsttests.
# (Wie das innen funktioniert, lernst du in V07 und V12. Du musst es nicht verstehen.)
import csv
import random

def messdatei_anlegen():
    rng = random.Random(15)
    with open("wellen.csv", "w", encoding="utf-8", newline="") as f:
        f.write("stichprobe,maschine,schicht,teil,durchmesser_mm\n")
        for s in range(1, 26):
            schicht = "Früh" if s <= 13 else "Spät"
            for maschine, mitte, sigma in (("D1", 20.000, 0.0040), ("D2", 20.006, 0.0050)):
                drift = 0.0012 * (s - 17) if (maschine == "D2" and s >= 18) else 0.0
                for t in range(1, 6):
                    d = rng.gauss(mitte + drift, sigma)
                    f.write(f"{s},{maschine},{schicht},{t},{d:.4f}\n")

messdatei_anlegen()

def _lese():
    with open("wellen.csv", encoding="utf-8") as f:
        return [(int(z[0]), z[1], z[2], int(z[3]), float(z[4])) for z in list(csv.reader(f))[1:]]

ROHDATEN = _lese()
OSG_T = 20.020
USG_T = 19.980

def _stats(werte):
    n = len(werte)
    m = sum(werte) / n
    s = (sum((x - m) ** 2 for x in werte) / (n - 1)) ** 0.5
    return m, s

def soll_cp_cpk(werte, usg=USG_T, osg=OSG_T):
    m, s = _stats(werte)
    return (osg - usg) / (6 * s), min(osg - m, m - usg) / (3 * s)

def pruefe(ist, soll, text):
    if ist == soll:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")

def pruefe_nah(ist, soll, text, toleranz=1e-9):
    # Für Kommazahlen: "gleich" heißt hier, die Abweichung ist winzig (Rundungsfehler!).
    try:
        gut = abs(float(ist) - soll) <= toleranz
    except (TypeError, ValueError):
        gut = False
    if gut:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")

def pruefe_ja(bedingung, text):
    # Für Diagramme und Tabellen: Die Bedingung wird aus den Objekten gebildet.
    if bedingung:
        print("OK: " + text)
    else:
        print("FEHLER: " + text)

print("wellen.csv angelegt:", len(ROHDATEN), "Messungen")

## Stufe 1 · Aufwärmen

### 1.1 Vorhersagen: Maske und Filter
Eine Series mit sechs Abweichungen vom Sollmaß in µm. **Schreib auf, was jede Zeile ausgibt**, bevor du die Zelle ausführst. Welche Zeile gibt eine Reihe aus True und False aus, welche eine kürzere Reihe, welche eine einzelne Zahl? Danach: Warum bekommt die letzte Zeile (auskommentiert) einen `ValueError`?

In [ ]:
# Meine Vorhersage: ...
import pandas as pd

abw = pd.Series([5, 12, -3, 7, -9, 0])         # Abweichung in µm
maske = abw > 4
print(maske)
print(abw[maske])
print(len(abw[maske]))
print(abw[maske].mean())
print(abw[(abw > 4) & (abw < 10)])
# print(abw[(abw > 4) and (abw < 10)])         # ValueError: Warum?

### 1.2 Die Datei ansehen
Bevor du etwas rechnest, schaust du, was in der Datei steht. Lies `wellen.csv` in die Tabelle `df`. Setze `zeilen` auf die Zahl der Zeilen, `spalten` auf die Zahl der Spalten und `namen` auf die Liste der Spaltennamen (`list(df.columns)`). Gib auch die ersten drei Zeilen aus.

<details><summary>Tipp anzeigen</summary>

Tipp: `df = pd.read_csv("wellen.csv")`. Die Form liefert `df.shape` (ohne Klammern), die Zeilenzahl auch `len(df)`. Die ersten Zeilen zeigt `df.head(3)`.
</details>

In [ ]:
import pandas as pd

df = None
zeilen = spalten = None
namen = None
# Dein Code hier

print(df.head(3) if df is not None else "df fehlt noch")

In [ ]:
pruefe_ja(df is not None and len(df) == 250, "df hat 250 Zeilen")
pruefe(zeilen, 250, "Zahl der Zeilen")
pruefe(spalten, 5, "Zahl der Spalten")
pruefe(namen, ["stichprobe", "maschine", "schicht", "teil", "durchmesser_mm"], "Spaltennamen")

### 1.3 Kennzahlen einer Spalte
Setze `mittel_alle`, `s_alle`, `kleinster` und `groesster` für die Spalte `durchmesser_mm` (alle 250 Teile beider Maschinen). Zur Kontrolle gib `df["durchmesser_mm"].describe()` aus: Findest du dieselben Zahlen?

<details><summary>Tipp anzeigen</summary>

Tipp: Die Spalte holst du mit `df["durchmesser_mm"]`. Darauf rechnen `mean()`, `std()`, `min()` und `max()`. `std()` ist die Stichproben-Standardabweichung (geteilt durch n − 1).
</details>

In [ ]:
mittel_alle = s_alle = kleinster = groesster = None
# Dein Code hier

print(mittel_alle, s_alle, kleinster, groesster)

In [ ]:
werte_alle = [z[4] for z in ROHDATEN]
m_, s_ = _stats(werte_alle)
pruefe_nah(mittel_alle, m_, "Mittelwert aller Teile", 1e-9)
pruefe_nah(s_alle, s_, "Standardabweichung aller Teile", 1e-9)
pruefe_nah(kleinster, min(werte_alle), "kleinster Wert", 1e-9)
pruefe_nah(groesster, max(werte_alle), "größter Wert", 1e-9)

### 1.4 Filtern: Teile der Spätschicht von D2
Zähle, wie viele Teile **von Maschine D2 in der Spätschicht** gemessen wurden. Baue die Maske mit `&` und Klammern, filtere die Tabelle (`spaet_d2`) und setze `anzahl = len(spaet_d2)`. Wie viele wären es mit `|` statt `&`? Probiere es aus.

<details><summary>Tipp anzeigen</summary>

Tipp: `df[(df["maschine"] == "D2") & (df["schicht"] == "Spät")]`. Jede Bedingung in Klammern, dazwischen `&`. Beim Text zwei Gleichheitszeichen und Anführungszeichen.
</details>

In [ ]:
spaet_d2 = None
anzahl = None
# Dein Code hier

print(anzahl)

In [ ]:
soll = len([z for z in ROHDATEN if z[1] == "D2" and z[2] == "Spät"])
pruefe(anzahl, soll, "Teile von D2 in der Spätschicht")
pruefe_ja(spaet_d2 is not None and set(spaet_d2["maschine"]) == {"D2"} and set(spaet_d2["schicht"]) == {"Spät"}, "die gefilterte Tabelle enthält nur D2 und Spät")

## Stufe 2 · Anwenden

### 2.1 Kennwerte je Maschine mit groupby
Berechne mit **einem** Ausdruck für jede Maschine Mittelwert, Standardabweichung und Anzahl der Spalte `durchmesser_mm` (`agg`) und speichere die Tabelle in `kennwerte`. Schau sie dir an: Welche Maschine streut stärker?

<details><summary>Tipp anzeigen</summary>

Tipp: `df.groupby("maschine")["durchmesser_mm"].agg(["mean", "std", "count"])`. Das Ergebnis hat zwei Zeilen (D1, D2) und drei Spalten.
</details>

In [ ]:
kennwerte = None
# Dein Code hier

print(kennwerte)

In [ ]:
pruefe_ja(kennwerte is not None and list(kennwerte.index) == ["D1", "D2"], "Zeilen D1 und D2")
pruefe_ja(kennwerte is not None and list(kennwerte.columns) == ["mean", "std", "count"], "Spalten mean, std, count")
for name in ("D1", "D2"):
    w = [z[4] for z in ROHDATEN if z[1] == name]
    m_, s_ = _stats(w)
    pruefe_ja(kennwerte is not None and abs(kennwerte.loc[name, "mean"] - m_) < 1e-9 and abs(kennwerte.loc[name, "std"] - s_) < 1e-9, name + ": Mittelwert und Standardabweichung")
    pruefe_ja(kennwerte is not None and kennwerte.loc[name, "count"] == len(w), name + ": Anzahl")

### 2.2 Cp und Cpk je Maschine
Rechne für beide Maschinen Cp und Cpk mit `OSG = 20.020` und `USG = 19.980`: `Cp = (OSG - USG) / (6 s)`, `Cpk = min(OSG - mittel, mittel - USG) / (3 s)`. Nimm Mittelwert und `s` aus `kennwerte` (2.1), oder filtere jede Maschine einzeln. Setze `cp_d1`, `cpk_d1`, `cp_d2`, `cpk_d2`. Welche Maschine ist fähig (Cpk ≥ 1,33)? Warum ist bei D2 der Cpk viel kleiner als der Cp?

<details><summary>Tipp anzeigen</summary>

Tipp: Mit `kennwerte.loc["D1", "mean"]` holst du einen einzelnen Wert. Oder: `d1 = df[df["maschine"] == "D1"]["durchmesser_mm"]`, dann `d1.mean()` und `d1.std()`. Das `min(...)` nimmt den kleineren der zwei Abstände zu den Grenzen.
</details>

In [ ]:
OSG = 20.020        # obere Spezifikationsgrenze in mm
USG = 19.980        # untere Spezifikationsgrenze in mm
cp_d1 = cpk_d1 = cp_d2 = cpk_d2 = None
# Dein Code hier

print(cp_d1, cpk_d1)
print(cp_d2, cpk_d2)

In [ ]:
for name, cp_, cpk_ in (("D1", cp_d1, cpk_d1), ("D2", cp_d2, cpk_d2)):
    scp, scpk = soll_cp_cpk([z[4] for z in ROHDATEN if z[1] == name])
    pruefe_nah(cp_, scp, name + ": Cp", 1e-9)
    pruefe_nah(cpk_, scpk, name + ": Cpk", 1e-9)
pruefe_ja(cpk_d1 is not None and cp_d1 is not None and cpk_d1 <= cp_d1 and cpk_d2 <= cp_d2, "Cpk ist nie größer als Cp")

### 2.3 Ausschussquote je Maschine
Lege eine neue Spalte `ausschuss` an, die `True` ist, wenn ein Teil außerhalb der Toleranz liegt (größer als OSG **oder** kleiner als USG). Der Mittelwert dieser Spalte je Maschine ist der Anteil: `quote` soll eine Reihe mit dem Anteil in **Prozent** für D1 und D2 sein.

<details><summary>Tipp anzeigen</summary>

Tipp: `df["ausschuss"] = (df["durchmesser_mm"] > OSG) | (df["durchmesser_mm"] < USG)`. Dann `df.groupby("maschine")["ausschuss"].mean() * 100`. Ein Teil genau auf der Grenze ist noch gut.
</details>

In [ ]:
quote = None
# Dein Code hier

print(quote)

In [ ]:
for name in ("D1", "D2"):
    w = [z[4] for z in ROHDATEN if z[1] == name]
    soll = 100 * len([x for x in w if x > OSG_T or x < USG_T]) / len(w)
    pruefe_nah(quote[name] if quote is not None else None, soll, "Quote " + name + " in Prozent", 1e-9)

### 2.4 Eine Funktion: cp_cpk
Cp und Cpk brauchst du für jede Maschine, jede Schicht, jedes Maß. Schreibe `cp_cpk(serie, usg, osg)`: Sie bekommt eine Spalte (Series) und die Grenzen und gibt das Paar `(cp, cpk)` zurück (ungerundet). Keine festen Zahlen im Rumpf, keine globale Tabelle.

<details><summary>Tipp anzeigen</summary>

Tipp: Im Rumpf: `mittel = serie.mean()`, `s = serie.std()`, dann die zwei Formeln, am Ende `return cp, cpk`. Alles, was du brauchst, steckt in den drei Parametern.
</details>

In [ ]:
def cp_cpk(serie, usg, osg):
    # Dein Code hier
    return None, None

import pandas as pd
d1 = df[df["maschine"] == "D1"]["durchmesser_mm"]
d2 = df[df["maschine"] == "D2"]["durchmesser_mm"]
print(cp_cpk(d1, 19.98, 20.02), cp_cpk(d2, 19.98, 20.02))

In [ ]:
import pandas as pd
for name, reihe in (("D1", d1), ("D2", d2)):
    scp, scpk = soll_cp_cpk(list(reihe))
    ist = cp_cpk(reihe, 19.98, 20.02)
    pruefe_ja(ist is not None and ist[0] is not None and abs(ist[0] - scp) < 1e-9 and abs(ist[1] - scpk) < 1e-9, name + ": (Cp, Cpk) stimmt")
r = pd.Series([10.01, 9.98, 10.03, 9.99, 10.00, 10.02])
scp, scpk = soll_cp_cpk(list(r), 9.9, 10.1)
ist = cp_cpk(r, 9.9, 10.1)
pruefe_ja(ist is not None and ist[0] is not None and abs(ist[0] - scp) < 1e-9 and abs(ist[1] - scpk) < 1e-9, "andere Grenzen (9,9 und 10,1): nichts ist fest eingebaut")

### 2.5 Regelkarte für Maschine D2
Zeichne die Regelkarte der Stichprobenmittel von D2: (a) Mittelwert je Stichprobe in `mittel`, (b) Mittellinie `mitte` und Streuung `sigma_x = s / √5` aus den **Stichproben 1 bis 15**, (c) Eingriffsgrenzen `oeg = mitte + 3 sigma_x` und `ueg = mitte - 3 sigma_x`, (d) die Nummern der Stichproben außerhalb als Liste in `auffaellig`. Zeichne auf `ax`: die Mittelwerte als Linie mit Punkten, drei Hilfslinien (`axhline`) mit unterschiedlichem Linienstil, Achsen mit Größe und Einheit, Titel.

<details><summary>Tipp anzeigen</summary>

Tipp: Wie im Labor-Beispiel: `d2 = df[df["maschine"] == "D2"]`, `mittel = d2.groupby("stichprobe")["durchmesser_mm"].mean()`, `anfang = d2[d2["stichprobe"] <= 15]`. Zeichnen: `ax.plot(mittel.index, mittel.values, marker="o")`, `ax.axhline(mitte, ...)`. Die Liste: `list(mittel[(mittel > oeg) | (mittel < ueg)].index)`.
</details>

In [ ]:
import matplotlib.pyplot as plt

mittel = mitte = sigma_x = oeg = ueg = None
auffaellig = None
fig, ax = plt.subplots(figsize=(8, 4))
# TODO: rechnen (a bis d), dann zeichnen und beschriften
plt.show()
print("Auffällig:", auffaellig)

In [ ]:
w2 = [z for z in ROHDATEN if z[1] == "D2"]
anf = [z[4] for z in w2 if z[0] <= 15]
m_, s_ = _stats(anf)
pruefe_nah(mitte, m_, "Mittellinie", 1e-9)
pruefe_nah(oeg, m_ + 3 * s_ / 5 ** 0.5, "obere Eingriffsgrenze", 1e-9)
pruefe_nah(ueg, m_ - 3 * s_ / 5 ** 0.5, "untere Eingriffsgrenze", 1e-9)
pruefe([int(x) for x in auffaellig] if auffaellig is not None else None, [20, 24, 25], "auffällige Stichproben")
pruefe_ja(len([l for l in ax.lines if len(set(float(v) for v in l.get_ydata())) > 1]) == 1, "eine Linie mit den Stichproben-Mittelwerten")
pruefe_ja(len([l for l in ax.lines if len(set(float(v) for v in l.get_ydata())) == 1]) >= 3, "drei waagerechte Hilfslinien")
pruefe_ja("Stichprobe" in ax.get_xlabel() and "mm" in ax.get_ylabel(), "Achsen mit Größe und Einheit")
pruefe_ja(ax.get_title() != "", "Titel vorhanden")

### 2.6 Fehler provozieren und lesen
Mach absichtlich je einen Fehler und lies die Meldung **von unten nach oben**: (a) `df["durchmesser"]` (falscher Spaltenname), (b) eine Maske mit `and` statt `&`, (c) `pd.read_csv("gibt_es_nicht.csv")`, (d) der Mittelwert einer Textspalte (`df["maschine"].mean()`). Welche Fehlerart bekommst du jeweils? Notiere sie in `fehlerarten`.

In [ ]:
import pandas as pd

fehlerarten = {"a": None, "b": None, "c": None, "d": None}   # z. B. "KeyError" als Text eintragen
# Dein Code hier: vier kleine Versuche (jeweils in try/except, damit das Notebook weiterläuft)
# Tipp: except Exception as fehler: print(type(fehler).__name__)

In [ ]:
pruefe(fehlerarten["a"], "KeyError", "a: falscher Spaltenname")
pruefe(fehlerarten["b"], "ValueError", "b: and statt &")
pruefe(fehlerarten["c"], "FileNotFoundError", "c: Datei nicht gefunden")
pruefe(fehlerarten["d"], "TypeError", "d: Mittelwert von Text")

### 2.7 Wiederholung: die Run-Regel als Schleife (V06)
Eine Regel der Regelkarte: **Liegen sechs Stichprobenmittel in Folge über der Mittellinie, hat sich der Prozess verschoben**, auch wenn kein Punkt außerhalb der Eingriffsgrenzen liegt. Schreibe eine Funktion `erste_serie(werte, grenze, laenge)` mit einer **Schleife** (keine pandas-Befehle): Sie zählt, wie viele Werte in Folge größer als `grenze` sind, und gibt die **Position** (ab 0) des Werts zurück, bei dem die Zahl `laenge` erreicht wird, sonst `None`. Wende sie auf die Stichprobenmittel von D2 an (`mittel` aus 2.5, Mittellinie `mitte`) und setze `erste` auf die **Stichprobennummer** dieses Werts.

<details><summary>Tipp anzeigen</summary>

Tipp: Zähler `folge = 0`. Für jeden Wert: Ist er größer als die Grenze, `folge = folge + 1`, sonst `folge = 0`. Sobald `folge == laenge`, `return position`. Die Position erhältst du mit `for position in range(len(werte))` und `werte[position]` (V06). Die Nummer der Stichprobe ist `mittel.index[position]`.
</details>

In [ ]:
def erste_serie(werte, grenze, laenge):
    # Dein Code hier
    return None

erste = None
# Dein Code hier: Funktion auf die Stichprobenmittel von D2 anwenden
print(erste)

In [ ]:
pruefe(erste_serie([5, 1, 6, 7, 8, 2], 4, 3), 4, "drei in Folge ab Position 2: Ende bei Position 4")
pruefe(erste_serie([5, 6, 7, 1], 4, 4), None, "nur drei in Folge: None")
pruefe(erste_serie([9], 4, 1), 0, "ein Wert, Länge 1")
pruefe(erste_serie([], 4, 2), None, "leere Liste")
pruefe(erste, 25, "bei D2 wird die Serie von sechs bei Stichprobe 25 erreicht")

## Stufe 3 · Knobeln

### 3.1 Maschine und Schicht gemeinsam: Wo liegt das Problem?
Gruppiere nach **zwei** Spalten (`groupby(["maschine", "schicht"])`) und berechne für jede der vier Kombinationen den Cpk. Setze `schlechteste` auf ein Tupel `(maschine, schicht)` mit dem kleinsten Cpk und `cpk_min` auf diesen Wert. Was sagt das über die Ursache (Maschine, Schicht oder beides)?

<details><summary>Tipp anzeigen</summary>

Tipp: Entweder mit einer Schleife über die vier Kombinationen und deiner Funktion aus 2.4, oder mit `.agg(["mean", "std"])` und einer Rechnung auf den Spalten. Die Gruppenschlüssel stehen danach als Zeilenbeschriftung (Index).
</details>

In [ ]:
schlechteste = None
cpk_min = None
# Dein Code hier

print(schlechteste, cpk_min)

In [ ]:
besten = {}
for m in ("D1", "D2"):
    for s in ("Früh", "Spät"):
        besten[(m, s)] = soll_cp_cpk([z[4] for z in ROHDATEN if z[1] == m and z[2] == s])[1]
soll_key = min(besten, key=besten.get)
pruefe(tuple(schlechteste) if schlechteste is not None else None, soll_key, "schlechteste Kombination")
pruefe_nah(cpk_min, besten[soll_key], "ihr Cpk", 1e-9)

### 3.2 Histogramm mit Spezifikationsgrenzen
Zeichne auf `ax` ein **Histogramm** (10 Klassen) der Durchmesser von Maschine D2 und markiere USG und OSG mit je einer senkrechten Linie (`axvline`) in unterschiedlichem Linienstil. Beschrifte die Achsen mit Größe und Einheit („Durchmesser d in mm“, „Anzahl Teile“), setze einen Titel und zähle in `ausser` die Teile von D2 außerhalb der Toleranz. Das Bild zeigt, warum der Cpk von D2 so klein ist.

<details><summary>Tipp anzeigen</summary>

Tipp: `ax.hist(d2["durchmesser_mm"], bins=10)`, dann `ax.axvline(USG, ...)` und `ax.axvline(OSG, ...)`. Zählen: `len(d2[(d2["durchmesser_mm"] > OSG) | (d2["durchmesser_mm"] < USG)])`.
</details>

In [ ]:
import matplotlib.pyplot as plt

ausser = None
fig, ax = plt.subplots()
# TODO: Histogramm, zwei Grenzlinien, Beschriftung, Titel; ausser zählen
plt.show()
print("Außerhalb:", ausser)

In [ ]:
pruefe(len(ax.patches), 10, "zehn Klassen")
pruefe_ja(len(ax.lines) == 2 and sorted(round(float(l.get_xdata()[0]), 3) for l in ax.lines) == [19.98, 20.02], "zwei senkrechte Linien bei 19,98 und 20,02")
pruefe_ja(len(ax.lines) == 2 and ax.lines[0].get_linestyle() != ax.lines[1].get_linestyle(), "die zwei Linien unterscheiden sich im Linienstil")
pruefe_ja("mm" in ax.get_xlabel() and "Anzahl" in ax.get_ylabel(), "Achsen mit Größe und Einheit")
pruefe_ja(ax.get_title() != "", "Titel vorhanden")
pruefe(ausser, len([z for z in ROHDATEN if z[1] == "D2" and (z[4] > OSG_T or z[4] < USG_T)]), "Teile von D2 außerhalb der Toleranz")

### 3.3 Deine eigene Datei
Lege eine eigene CSV-Datei (Messdaten aus deinem Betrieb, aus einer phyphox-Messung oder dem Projekt) in den Ordner des Notebooks und trage den Namen ein. Schau sie zuerst **roh** an (`open(...).read()[:300]`): Trennzeichen? Dezimalzeichen? Kopfzeile? Lies sie dann mit `pd.read_csv` und den passenden `sep=` und `decimal=`, prüfe `df.shape` und `df.columns`, rechne Mittelwert und Standardabweichung der wichtigsten Spalte und zeichne ein Diagramm mit Achsentiteln (Checkliste V14). Notiere in einer Markdown-Zelle, was dir auffällt und welche Teilmenge du zusätzlich auswerten möchtest.

In [ ]:
dateiname = "wellen.csv"     # hier deine Datei eintragen
# Dein Code hier

## Rückblick
- `pd.read_csv("datei.csv")` lädt eine Tabelle, bei deutschen Dateien mit `sep=";"` und `decimal=","`. **Gleich danach** `shape` und `columns` ansehen.
- `df["spalte"]` ist eine Series (`mean`, `std`, `min`, `max`, `describe`), `df[["a", "b"]]` eine Tabelle mit zwei Spalten.
- Eine **Maske** (`df["x"] > 5`) ergibt True/False je Zeile, `df[maske]` filtert. Bedingungen mit `&`, `|`, `~` und Klammern verknüpfen, nie `and`/`or`/`not`.
- `groupby("spalte")["wert"].agg([...])` rechnet je Gruppe. Jede Kennzahl gehört in die **richtige Teilmenge** (eine Maschine, eine Schicht).
- Cp = (OSG − USG) / (6 s), Cpk = min(OSG − m, m − USG) / (3 s), nie größer als Cp. Eingriffsgrenzen der Regelkarte (m ± 3 s/√n) sind **nicht** die Toleranzgrenzen.
- KI-Auswertungen prüfst du auf: richtige Teilmenge, `min` statt `max` im Cpk, `&` statt `and`, Eingriffsgrenzen statt Toleranzgrenzen, und vergleichst die Zahl mit einer Handrechnung.